# Lyric-Music Congruence — Pipeline

End-to-end, **resumable** pipeline. Every stage skips work that is already done and
persists progress to `data/project.db`, so you can stop and restart at any point.

**Source of truth:** the LRCLIB dump (songs with time-synced lyrics). Audio,
popularity, mood and embeddings all key off the LRCLIB track ID.

Stages: `setup → sample → audio → popularity → mood → chorus → embeddings → alignment → combine`.

Prerequisites: install `requirements.txt`, place `lrclib-db-dump-*.sqlite3` under `data/`
(or set `LRCLIB_DUMP`), and export `SPOTIFY_CLIENT_ID` / `SPOTIFY_CLIENT_SECRET`
(plus optional `YOUTUBE_API_KEY`, `LASTFM_API_KEY`).

In [ ]:
import sys, logging
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))   # make `lmc` importable

from lmc import (config, lrclib, audio, popularity, mood,
                 chorus, embeddings, alignment, combine, db)
from lmc.utils import setup_logging
setup_logging(logging.INFO)

config.ensure_dirs()
print(config.summary())

## 1 — Setup & sample

`setup()` reports how many synced-lyric songs exist in LRCLIB (the *universe*),
how many we've already sampled, and roughly how many remain. The first call runs
a one-time full count (cached afterwards). Then pick a **session target** and
`sample()` randomly draws that many *new* songs into the corpus.

In [ ]:
status = lrclib.setup()
status

In [ ]:
TARGET = 50          # how many NEW songs to add this session
new_ids = lrclib.sample(TARGET, seed=None)
print(f'Added {len(new_ids)} songs; corpus now {lrclib.setup()["sampled"]}.')

## 2 — Audio (official audio, not music videos)

Downloads `data/audio/<track_id>.mp3` via yt-dlp, preferring auto-generated
“- Topic” channels and audio uploads over music videos. Captures YouTube
view / like / comment counts. Pass `limit=` to cap a session.

In [ ]:
audio.download_pending(limit=None)

## 3 — Popularity & genre recovery

Spotify popularity + artist genre tags (→ recovered genre / orientation), Deezer
rank, optional Last.fm, and the YouTube metrics mirrored from stage 2.

In [ ]:
popularity.fetch_pending(limit=None)

## 4 — Mood (librosa proxies) & 5 — Chorus detection

Mood needs audio; chorus detection is lyrics-only (runs on the whole corpus).

In [ ]:
mood.extract_pending(limit=None)
chorus.compute_pending()

## 6 — Embeddings (MuQ-MuLan + LAION-CLAP)

Computes and **caches** a per-song embedding bundle (`data/embeddings/<model>/<id>.npz`)
covering song / line-window / segment levels. Only songs without a cached bundle
are processed — embeddings are never recomputed. First run downloads model weights.

In [ ]:
embeddings.embed_pending('mulan', limit=None)
embeddings.embed_pending('clap',  limit=None)

## 7 — Alignment (LMC) & 8 — Combine

Turns cached embeddings into LMC values (song / line-windows / chorus vs.
non-chorus) and assembles the analysis tables in `results/`.

In [ ]:
alignment.compute_pending()
combine.build_master()

In [ ]:
import pandas as pd
pd.read_csv(config.RESULTS_DIR / 'master_results.csv').head()

## Progress dashboard

Re-run any time to see how far each stage has gotten. Safe to interrupt and
resume: rerun the cells above and only the unfinished songs are processed.

In [ ]:
db.progress_report()

## Next: statistics & modeling (R)

```bash
Rscript analysis/summary_stats.R        # quick descriptives + figures
Rscript stan/run_models.R mulan 500 1   # fit models on a random N=500 sample
```
`run_models.R` exposes `sample_corpus(df, N, seed)` to fit on a random subset of
whatever has been gathered so far.